# 02-Stateful AI UIs (Streamlit)

In Lesson 01, we used Gradio to rapidly wrap Python functions into Web UIs. Gradio is excellent for quick input/output translations. However, if you look at modern AI products (like ChatGPT, Midjourney's web interface, or enterprise AI dashboards), they are not just single functions. They are **Stateful Applications** with complex layouts, sidebars, data visualizations, and continuous chat histories.

To build production-grade, highly custom AI frontends entirely in Python (without writing a single line of React, HTML, or CSS), the industry standard is **Streamlit**.

Let's set up our environment to master the physics of Stateful Web Rendering.

In [3]:
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel
import streamlit as st
import time
import seaborn as sns

# Set professional visualization styling
sns.set_theme(style="whitegrid")

print("✅ PyTorch & Streamlit Stateful UI Environment Ready.")

✅ PyTorch & Streamlit Stateful UI Environment Ready.


# 1. The Physics of the Execution Loop (Top-to-Bottom)

To use Streamlit without crashing your server, you must fundamentally understand its rendering architecture. It is completely different from Gradio, Flask, or FastAPI.

**The Streamlit Paradigm:**
Every single time a user interacts with a widget on the screen (e.g., clicking a button, typing in a text box, or dragging a slider), **Streamlit completely destroys the application and re-runs the entire Python script from line 1 to the very last line.**

### The Catastrophic AI Danger

If you write standard Python code to load a 10-Gigabyte LLM into VRAM:
`model = LlamaForCausalLM.from_pretrained("llama-3")`
And then you put a "Submit" button below it...
Every time the user clicks "Submit", Streamlit will re-execute line 1. It will attempt to load the 10GB model into VRAM *again*, instantly causing an Out-Of-Memory (OOM) fatal crash.

# 2. Protecting the GPU (`@st.cache_resource`)

To prevent the top-to-bottom execution loop from destroying our GPU memory, we must use **Caching**.

We wrap our heavy PyTorch initialization logic inside a function, and we decorate it with `@st.cache_resource`.
When Streamlit runs from top-to-bottom on Step 1, it executes the function, loads the model into VRAM, and mathematically stores a pointer to that specific memory address in the global cache.
When the user clicks a button and the script re-runs, Streamlit sees the `@st.cache_resource` decorator, completely skips the model-loading code, and instantly retrieves the pointer from the cache.

# 3. The Mathematics of Memory (`st.session_state`)

Because the script re-runs on every click, **standard Python variables are completely erased.**
If you have a variable `chat_history = []`, and the user types *"Hello"*, you append it: `chat_history.append("Hello")`.
But the moment they type their second message, the script re-runs. Line 1 executes: `chat_history = []`. The memory of "Hello" is violently overwritten and destroyed.

To build an AI Assistant, we must inject a persistent mathematical ledger that survives the execution loop. This is called **Session State**.
`st.session_state` is a global dictionary that persists across re-runs for a specific user session. If you store the chat history inside `st.session_state.messages`, the UI will remember the conversation infinitely.

# 4. Architecting an Enterprise ChatGPT Clone in PyTorch & Streamlit

Let's combine these concepts to architect a production-ready conversational AI UI. We will cache a pre-trained LLM, initialize the Session State, render the historical UI, and yield the tokens using Streamlit's native `st.chat_message` blocks.

*(Note: In a real environment, you run Streamlit files from the terminal using `streamlit run app.py`. For this notebook, we will architect the exact code you would put in `app.py`)*

In [4]:
# --- 🏗️ app.py ---
import streamlit as st
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel
import time

# 1. Page Configuration
st.set_page_config(page_title="Enterprise AI Assistant", page_icon="🤖", layout="centered")
st.title("🧠 Neural Chat Interface")
st.markdown("Powered by PyTorch, Transformers, and Streamlit Session State.")

# 2. Protect the GPU (Caching the Foundation Model)
# We use cache_resource for non-serializable objects like PyTorch models or Database connections!
@st.cache_resource
def load_foundation_model():
    print("Executing heavy VRAM initialization... (This will only print ONCE!)")
    tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
    model = GPT2LMHeadModel.from_pretrained("gpt2")
    model.eval()
    return tokenizer, model

tokenizer, model = load_foundation_model()

# 3. Initialize the Mathematical Ledger (Session State)
# If the user just arrived, 'messages' won't exist. We initialize it to an empty array.
if "messages" not in st.session_state:
    st.session_state.messages = []

# 4. Render the Historical Context
# Streamlit erased the screen when it re-ran. We must re-draw the entire chat history.
for message in st.session_state.messages:
    # st.chat_message creates a beautiful visual block with an avatar
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

# 5. Handle the User Interaction Trigger
# st.chat_input creates the text box at the bottom of the screen. 
# It pauses execution here until the user hits Enter.
if prompt := st.chat_input("Ask the model a question..."):
    
    # A. Display the User's prompt instantly
    with st.chat_message("user"):
        st.markdown(prompt)
        
    # B. Save the User's prompt to the persistent memory ledger
    st.session_state.messages.append({"role": "user", "content": prompt})
    
    # C. Execute the Autoregressive AI Generation
    with st.chat_message("assistant"):
        # We create an empty placeholder in the UI that we can dynamically update
        message_placeholder = st.empty()
        full_response = ""
        
        # Format the prompt for the model
        input_ids = tokenizer.encode(prompt, return_tensors="pt")
        
        # NOTE: For true production streaming, we would use HuggingFace's TextIteratorStreamer.
        # To keep the physics clear, we will simulate the streaming delay of the final output.
        with torch.no_grad():
            output_ids = model.generate(input_ids, max_length=50, pad_token_id=tokenizer.eos_token_id)
            generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
            
            # Extract just the AI's response (removing the prompt)
            ai_response = generated_text[len(prompt):].strip()
            
        # Simulate the physical token-by-token streaming effect
        for chunk in ai_response.split():
            full_response += chunk + " "
            time.sleep(0.05) # Simulated GPU computation time per token
            # Dynamically overwrite the placeholder to create the typing illusion!
            message_placeholder.markdown(full_response + "▌")
            
        # Finalize the response (Remove the blinking cursor)
        message_placeholder.markdown(full_response)
        
    # D. Save the AI's response to the persistent memory ledger
    st.session_state.messages.append({"role": "assistant", "content": full_response})

print("Success! If executed via `streamlit run`, this code generates a flawless, stateful ChatGPT replica.")

2026-06-16 15:00:07.728 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-16 15:00:07.730 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-16 15:00:07.854 
  command:

    streamlit run /root/micromamba/envs/ds/lib/python3.11/site-packages/ipykernel_launcher.py [ARGUMENTS]
2026-06-16 15:00:07.855 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-16 15:00:07.856 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-16 15:00:07.857 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-16 15:00:07.858 Thread 'MainThread': missing ScriptRunContext! This warning can be ignore

Executing heavy VRAM initialization... (This will only print ONCE!)


2026-06-16 15:00:22.712 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-16 15:00:22.713 Session state does not function when running a script without `streamlit run`
2026-06-16 15:00:22.714 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-16 15:00:22.715 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-16 15:00:22.716 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-16 15:00:22.717 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-16 15:00:22.717 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-16 15:00:22.718 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-16 15:00

Success! If executed via `streamlit run`, this code generates a flawless, stateful ChatGPT replica.


## Real-World Use Case or Analogy:

Think of the Streamlit execution model like **A Waiter with Amnesia (Groundhog Day)**:

* **The Problem (Top-to-Bottom Execution)**: You walk into a restaurant and sit down. The waiter walks up, introduces himself, brings you a menu, and asks for your drink order. You say *"Water"*. The waiter turns around, immediately suffers absolute amnesia, walks back up to you, introduces himself, brings you a menu, and asks for your drink order. You are trapped in an infinite loop because the waiter forgets everything the second he stops talking to you.
* **The GPU Crash (`@st.cache_resource`)**: Worse, every time the waiter walks up to you, he goes into the kitchen, slaughters a cow, and builds a brand new oven from scratch just in case you order a steak. The kitchen (VRAM) goes bankrupt in 5 minutes. You fix this by telling the waiter, *"Look in the kitchen first. If the oven is already built, DO NOT build another one."* (Caching).
* **The Solution (`st.session_state`)**: You give the waiter a permanent notebook that he keeps in his pocket. When you say *"Water"*, he writes it down in the notebook (Session State). Now, when his amnesia triggers, he walks up to you, introduces himself, but checks his notebook and says: *"I see you already ordered Water. What would you like for dinner?"* The illusion of continuous conversation is perfectly preserved!